In [1]:
import sys
import os
notebook_dir = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..', 'lime_ndt')))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..')))

## California Housing Dataset

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# LIME
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeTabularExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0
        return self

# ========================
# Charger dataset complet
# ========================
data = fetch_california_housing()
X, y = data.data, data.target
feature_names = data.feature_names
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (MLP)
# ========================
mlp_global = MLPRegressor(hidden_layer_sizes=(128, 64, 32),
                          activation='relu',
                          solver='adam',
                          max_iter=2000,
                          random_state=42)
mlp_global.fit(X_train, y_train)
predict_fn = mlp_global.predict

# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train, feature_names=feature_names,
    discretize_continuous=False, mode='regression'
)
explainer_ndt = LimeNDTExplainer(
    X_train, feature_names=feature_names,
    discretize_continuous=False, mode='regression'
)

# ========================
# Fonction d'explication
# ========================
def get_explanation_vector(explainer, instance, predict_fn, local_model):
    exp = explainer.explain_instance(instance, predict_fn, model_regressor=local_model)
    weights = dict(exp.local_exp[1])
    return np.array([weights.get(i, 0.0) for i in range(len(feature_names))])

# ========================
# Fonction de régularité
# ========================
def compute_regularity(explainer, local_model_cls, X_data, k=5, save_path=None):
    n = len(X_data)
    E = np.zeros((n, X_data.shape[1]))

    print(f"→ Génération des {n} explications locales...")
    for i, x in enumerate(tqdm(X_data)):
        try:
            E[i] = get_explanation_vector(explainer, x, predict_fn, local_model_cls())
        except Exception as e:
            # si LIME échoue sur une instance, mettre un vecteur nul
            print(f"⚠️ Instance {i} skipped ({e})")
            E[i] = np.zeros(X_data.shape[1])

    # Remplacer NaN et inf par 0 pour éviter les erreurs de similarité
    E = np.nan_to_num(E, nan=0.0, posinf=0.0, neginf=0.0)

    if save_path:
        np.save(save_path, E)
        print(f"Explanations saved to {save_path}")

    print("→ Calcul des similarités cosinus entre voisins...")
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(X_data)
    _, indices = nbrs.kneighbors(X_data)
    indices = indices[:, 1:]

    cos_sims = []
    for i, neigh_idx in enumerate(indices):
        # Extra safety: handle any all-zero vectors
        Ei = E[i].reshape(1, -1)
        En = E[neigh_idx]
        if np.all(Ei == 0) or np.all(En == 0):
            continue
        sims = cosine_similarity(Ei, En)[0]
        cos_sims.append(np.mean(sims))

    return np.mean(cos_sims) if len(cos_sims) > 0 else 0.0


# ========================
# Exécution sur tout X_test
# ========================
results = {}
results["LinearRegression"] = compute_regularity(explainer_classic, LinearRegression, X_test)
results["DecisionTree"] = compute_regularity(explainer_ndt, DecisionTreeWrapper, X_test)
results["NDT"] = compute_regularity(
    explainer_ndt,
    lambda: NDTRegressorWrapper(D=X_train.shape[1], gammas=[100, 1]),
    X_test
)

print("\n=== Régularité moyenne sur tout le jeu de test ===")
for model_name, score in results.items():
    print(f"{model_name}: {score:.3f}")


→ Génération des 5160 explications locales...


100%|██████████| 5160/5160 [01:04<00:00, 80.38it/s]


→ Calcul des similarités cosinus entre voisins...
→ Génération des 5160 explications locales...


100%|██████████| 5160/5160 [07:28<00:00, 11.52it/s]


→ Calcul des similarités cosinus entre voisins...


ValueError: Input contains NaN.